# Day 1 — AI Football Discussion Generator

**Project:** Football Adda  
**Scenario:** Messi vs Ronaldo — Who is the greatest?  
**Goal:** Understand Python data structures, functions, JSON, prompts, and then connect an LLM.

This notebook first loads a complete ready-made two-minute Bangla script. Then you will build the same structure yourself and prepare a prompt for a local LLM.

## 1. Imports and project setup

`json` is part of Python's standard library. You do not need to install it.

In [1]:
import json
from pathlib import Path

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

print("Project is ready.")

Project is ready.


## 2. Define the three fictional presenters

Each presenter is a Python dictionary. The list contains all three dictionaries.

In [2]:
presenters = [{'name': 'Rahim', 'role': 'Host', 'personality': 'Professional, curious, balanced and confident', 'speaking_style': 'Asks direct questions, controls the discussion and gives everyone a chance', 'position': 'Neutral moderator'}, {'name': 'Karim', 'role': 'Football Fan', 'personality': 'Funny, emotional, energetic and passionate', 'speaking_style': 'Uses humour, emotional arguments and passionate examples', 'position': 'Strong Messi supporter'}, {'name': 'Jamal', 'role': 'Football Analyst', 'personality': 'Logical, analytical, calm but argumentative', 'speaking_style': 'Uses tactical reasoning, comparisons and evidence; challenges exaggerations', 'position': 'Strong Ronaldo supporter'}]

for presenter in presenters:
    print(presenter['name'], '-', presenter['role'])

Rahim - Host
Karim - Football Fan
Jamal - Football Analyst


## 3. Load the complete sample discussion JSON

Keep `ai_football_discussion.json` in the same folder as this notebook, or update the path below.

In [3]:
json_file = Path("ai_football_discussion.json")

with json_file.open("r", encoding="utf-8") as file:
    discussion = json.load(file)

print("Title:", discussion["title"])
print("Topic:", discussion["topic"])
print("Number of dialogue turns:", len(discussion["conversation"]))

Title: মেসি বনাম রোনালদো: সর্বকালের সেরা কে?
Topic: মেসি বনাম রোনালদো
Number of dialogue turns: 22


## 4. Display the discussion like a TV script

Notice how each dialogue turn has a speaker, emotion, and dialogue. This structure will later help us select voices and animate facial expressions.

In [4]:
def show_conversation(conversation):
    for index, line in enumerate(conversation, start=1):
        speaker = line["speaker"]
        emotion = line["emotion"]
        dialogue = line["dialogue"]

        print(f"{index:02d}. {speaker} [{emotion}]")
        print(dialogue)
        print()

show_conversation(discussion["conversation"])

01. Rahim [welcoming]
স্বাগতম ‘ফুটবল আড্ডা’ অনুষ্ঠানে। আজকের প্রশ্নটা এমন, যেটা নিয়ে বন্ধুদের আড্ডা কখনো শেষ হয় না—সর্বকালের সেরা কে, লিওনেল মেসি নাকি ক্রিস্টিয়ানো রোনালদো? করিম ভাই, আপনি তো প্রস্তুত হয়েই এসেছেন!

02. Karim [excited]
প্রস্তুত না হয়ে উপায় আছে? মেসির নাম শুনলেই আমার ফুটবলীয় হৃদয় নাচতে শুরু করে! আমার কাছে মেসি শুধু গোল করেন না, পুরো খেলাটাকেই সুন্দর করে তোলেন।

03. Jamal [amused]
হৃদয় নাচলে ভালো, কিন্তু মাঠে তো পা দিয়ে খেলতে হয়! রোনালদোর শারীরিক সক্ষমতা, গোল করার ক্ষুধা, হেডিং আর বিভিন্ন লিগে নিজেকে প্রমাণ করার ক্ষমতা অসাধারণ।

04. Rahim [curious]
দুজনেরই যুক্তি আছে। কিন্তু একটা বিষয় পরিষ্কার করি—আমরা কি শুধু গোলের সংখ্যা দিয়ে সেরা খেলোয়াড় ঠিক করব, নাকি দলের খেলায় অবদানও বিবেচনা করব?

05. Karim [confident]
অবশ্যই পুরো খেলাটা দেখতে হবে। মেসি গোল করেন, আবার সতীর্থদের জন্য সুযোগ তৈরি করেন, পাস দিয়ে ডিফেন্স ভেঙে দেন। তিনি একই সঙ্গে গোলদাতা এবং প্লেমেকার।

06. Jamal [challenging]
সেটা ঠিক। তবে রোনালদোও শুধু গোলমেশিন নন। তিনি জায়গা বুঝে দৌড়ান, চাপের মুহূর্তে দায়িত্ব নেন এবং বড় 

## 5. Understand the approximate speaking duration

A two-minute script depends on speaking speed and pauses. This helper estimates duration using a configurable words-per-minute rate. For Bangla, this is only a rough estimate because written word counts do not perfectly predict spoken duration.

In [5]:
def estimate_duration_seconds(conversation, words_per_minute=130):
    total_words = sum(
        len(line["dialogue"].split())
        for line in conversation
    )
    minutes = total_words / words_per_minute
    return total_words, round(minutes * 60)

word_count, estimated_seconds = estimate_duration_seconds(
    discussion["conversation"]
)

print("Approximate whitespace-separated word count:", word_count)
print("Estimated duration in seconds:", estimated_seconds)
print("Estimated duration in minutes:", round(estimated_seconds / 60, 2))

Approximate whitespace-separated word count: 443
Estimated duration in seconds: 204
Estimated duration in minutes: 3.4


## 6. Create a reusable prompt

A prompt tells the LLM what role to play, what topic to discuss, which characters to use, and what output format to return.

In [6]:
def create_prompt(topic, presenters, target_seconds=120):
    presenter_instructions = []

    for presenter in presenters:
        details = (
            f"Name: {presenter['name']}\n"
            f"Role: {presenter['role']}\n"
            f"Personality: {presenter['personality']}\n"
            f"Speaking style: {presenter['speaking_style']}\n"
            f"Position: {presenter['position']}"
        )
        presenter_instructions.append(details)

    presenter_text = "\n\n".join(presenter_instructions)

    prompt = f'''
You are a professional Bangla football TV discussion scriptwriter.

Show name: Football Adda
Topic: {topic}
Target spoken duration: approximately {target_seconds} seconds.

Presenters:
{presenter_text}

Requirements:
1. Write natural, conversational Bangla.
2. Keep each presenter's personality and viewpoint consistent.
3. Include a host introduction, debate, disagreement, humour and conclusion.
4. Let all three presenters speak several times.
5. Make the discussion sound like a real TV panel, not an essay.
6. Do not invent statistics, match results or factual claims.
7. Aim for approximately two minutes of spoken dialogue.
8. Return valid JSON only, with this structure:
{{
  "show_name": "Football Adda",
  "title": "A Bangla title",
  "topic": "{topic}",
  "language": "bn",
  "target_duration_seconds": 120,
  "presenters": [],
  "conversation": [
    {{
      "speaker": "Rahim",
      "emotion": "curious",
      "dialogue": "Bangla dialogue"
    }}
  ]
}}
'''
    return prompt

topic = "মেসি বনাম রোনালদো: সর্বকালের সেরা কে?"
prompt = create_prompt(topic, presenters)

print(prompt)


You are a professional Bangla football TV discussion scriptwriter.

Show name: Football Adda
Topic: মেসি বনাম রোনালদো: সর্বকালের সেরা কে?
Target spoken duration: approximately 120 seconds.

Presenters:
Name: Rahim
Role: Host
Personality: Professional, curious, balanced and confident
Speaking style: Asks direct questions, controls the discussion and gives everyone a chance
Position: Neutral moderator

Name: Karim
Role: Football Fan
Personality: Funny, emotional, energetic and passionate
Speaking style: Uses humour, emotional arguments and passionate examples
Position: Strong Messi supporter

Name: Jamal
Role: Football Analyst
Personality: Logical, analytical, calm but argumentative
Speaking style: Uses tactical reasoning, comparisons and evidence; challenges exaggerations
Position: Strong Ronaldo supporter

Requirements:
1. Write natural, conversational Bangla.
2. Keep each presenter's personality and viewpoint consistent.
3. Include a host introduction, debate, disagreement, humour an

## 7. Optional: connect to a local LLM using Ollama

First install Ollama on your computer and download a model that your hardware can run. Then install the Python package from a notebook cell:

```python
%pip install ollama
```

Example model name below is only an example. It must already be pulled in Ollama. Local model quality for Bangla varies, so review the generated script.

In [7]:
# Uncomment and run after installing Ollama and pulling a model.
#
# import ollama
#
# def generate_discussion_with_ollama(topic, presenters, model="qwen2.5:3b"):
#     prompt = create_prompt(topic, presenters)
#
#     response = ollama.chat(
#         model=model,
#         messages=[{"role": "user", "content": prompt}],
#         format="json"
#     )
#
#     return json.loads(response["message"]["content"])
#
# generated_discussion = generate_discussion_with_ollama(topic, presenters)
# show_conversation(generated_discussion["conversation"])

## 8. Validate the LLM output

Never assume generated JSON is correct. Validate required fields and allowed speaker names before using it.

In [8]:
def validate_discussion(data):
    required_top_level = {"title", "topic", "conversation"}
    missing = required_top_level - data.keys()

    if missing:
        raise ValueError(f"Missing top-level fields: {missing}")

    allowed_speakers = {p["name"] for p in presenters}

    if not isinstance(data["conversation"], list) or not data["conversation"]:
        raise ValueError("conversation must be a non-empty list")

    for index, line in enumerate(data["conversation"], start=1):
        for field in ("speaker", "emotion", "dialogue"):
            if field not in line:
                raise ValueError(f"Turn {index} is missing: {field}")

        if line["speaker"] not in allowed_speakers:
            raise ValueError(f"Unknown speaker in turn {index}: {line['speaker']}")

        if not isinstance(line["dialogue"], str) or not line["dialogue"].strip():
            raise ValueError(f"Empty dialogue in turn {index}")

    return True

validate_discussion(discussion)
print("Sample discussion passed validation.")

Sample discussion passed validation.


## 9. Save a JSON file

This saves the ready-made discussion. If you generate a new discussion with an LLM, replace `discussion` with that generated result.

In [9]:
output_file = DATA_DIR / "football_discussion_output.json"

with output_file.open("w", encoding="utf-8") as file:
    json.dump(discussion, file, ensure_ascii=False, indent=2)

print("Saved to:", output_file.resolve())

Saved to: /home/spectrum-se/AI-ML/ai_football_studio_day1/data/football_discussion_output.json


## 10. Your exercises

1. Change the topic to **বাংলাদেশ ফুটবলের ভবিষ্যৎ**.
2. Change Karim's personality to be a Brazil supporter.
3. Add a fourth presenter named Nabila, a women's football analyst.
4. Update the prompt so the LLM includes Nabila.
5. Generate a new script and validate its JSON.
6. Listen to the script yourself and adjust the target duration if needed.

**Important:** Today we are not training an LLM from scratch. We are learning how to prepare structured input, instruct a pretrained model, validate its response, and save the result. Fine-tuning is a later milestone.